# Predictions

Here we finally deploy our trained models to make predictions on real images.
The program uses Sliding Window method to extract sub-images from the full satellite image.
It then extracts features of those sub-images to determine the presence of waste in them.

In [2]:
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import time
import Utilities

In [11]:
src_folder_path = '/Users/sinner/Desktop/images_to_scan'
dest_folder_path = '/Users/sinner/Desktop/scanning_output'

In [4]:
# Defining a custom standard scaler to scale features

mean_std_df = pd.read_csv('mean_std_df.csv')

def my_standard_scaler(features):
    scaled_features = []
    for i, value in enumerate(features):
        mean = mean_std_df.iat[i, 1]
        std = mean_std_df.iat[i, 2]
        scaled_features.append((value - mean) / std)
    return np.array(scaled_features)

In [5]:
# Load model
loaded_model = pickle.load(open('trained_models/RandomForestClassifier.sav', 'rb'))
print('Model Loaded')

Model Loaded


In [10]:
# Driver code

# iterating in source folder
for filename in os.listdir(src_folder_path):

    # Check whether filetype is correct
    if not filename.endswith('.jpg'):
        continue

    print(f"Scanning {filename}")

    # load image
    img = cv2.imread(os.path.join(src_folder_path, filename))

    positiveWindows = []
    for window_coord in Utilities.get_window_coords(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), 100, 100):

        # get cropped window from img
        # cropped_img = img[top:bottom, left:right]
        window = img[window_coord[0]:window_coord[1], window_coord[2]:window_coord[3]]

        # get features of window
        features = Utilities.get_features(window)
        # Feature Scaling
        features = my_standard_scaler(features)

        # Predict class using model
        predicted_class = loaded_model.predict(np.reshape(features, (1,-1)))
        # print(predicted_class)

        if predicted_class[0] == 1:
            # append coordinates into positiveWindows list if waste detected
            positiveWindows.append(window_coord)

        overlay_time_start = time.time()
        for window_coord in positiveWindows:
            # Overlay the bounding box on the image
            cv2.rectangle(img, (window_coord[2], window_coord[0]), (window_coord[3], window_coord[1]), (0, 255, 0), 2)

    cv2.imwrite(os.path.join(dest_folder_path, filename), img)

Scanning 0515.jpg
Scanning 0514.jpg
Scanning 0516.jpg
Scanning 0113.jpg
Scanning 0517.jpg
Scanning 0513.jpg
Scanning 0116.jpg
Scanning 0512.jpg
Scanning 0510.jpg
Scanning 0114.jpg
Scanning 0115.jpg
Scanning 0511.jpg
Scanning 0006.jpg
Scanning 0520.jpg
Scanning 0509.jpg
Scanning 0519.jpg
Scanning 0518.jpg
